# TARANG: Dispersion Mapping (choosing H and W)

This notebook walks through the dispersion-mapping tools that choose a waveguide
cross-section (core height *H*, width *W*) for a Kerr comb: anomalous dispersion with a
single guided TE mode, robust to fabrication errors. All local, no credits; about 15–30
minutes on a laptop. Results go to `results/` next to this notebook.

| Step | Tool | Answers |
|---|---|---|
| 1 | `solve_point` | What are *D*, *n*<sub>g</sub> and the mode counts of one cross-section? |
| 2 | `sweep` + `maps` | Where in *H* × *W* is the target region? |
| 3 | `convergence` | Is the mesh fine enough for these numbers? |
| 4 | `sweep --points --tol` + `report` | Does a finalist stay in the target region within the fabrication tolerance? |
| 5 | `broadband` | How does *D* vary across the band; what is the ring's integrated dispersion? |

The grids here are tiny so the notebook runs quickly; real maps use the larger grids in
the [tool README](../tarang/waveguide/dispersion_mapping/README.md). Set the wavelength
and stack once in the next cell; every step uses them.

In [ ]:
# ---- Settings used by every step ---- #
import subprocess
import sys
from pathlib import Path

WL = 1.55                        # centre wavelength, um (e.g. 0.78 for the 780 nm platform)
CORE, SUBSTRATE, CLADDING = "si3n4", "sio2", "air"     # keys in tarang/media/media.json
STEPS = 50                       # mesh steps per wavelength: >= 50 for maps, 70 for decisions


def tool(name, *args):
    """Run python -m tarang.waveguide.dispersion_mapping.<name> with this kernel's Python."""
    cmd = [sys.executable, "-m", f"tarang.waveguide.dispersion_mapping.{name}",
           *map(str, args)]
    out = subprocess.run(cmd, capture_output=True, text=True)
    print("\n".join(l for l in out.stdout.splitlines() if l.strip()))
    if out.returncode:
        print(out.stderr[-3000:])
        raise RuntimeError(f"{name} failed")


STACK = ["--core", CORE, "--substrate", SUBSTRATE, "--cladding", CLADDING]

## 1. One cross-section

`solve_point(H, W)` solves the modes over a small band around the wavelength, tracks TE00
and TM00, and fits *n*<sub>eff</sub>(λ) to get *D*, β₂ and *n*<sub>g</sub>. It also
counts the guided TE and TM modes (single-mode check).

In [ ]:
# ---- Step 1: one point ---- #
from tarang.waveguide.dispersion_mapping import solve_point

r = solve_point(0.70, 1.00, wl0=WL, steps=STEPS, core=CORE, substrate=SUBSTRATE, cladding=CLADDING)
print(f"D(TE00) = {r['D_TE0']:.1f} ps/(nm km),  n_g = {r['ng_TE0']:.4f},  "
      f"A_eff = {r['Aeff_TE0_um2']:.3f} um^2")
print(f"guided TE / TM modes: {r['guided_TE']} / {r['guided_TM']}")

## 2. Map the design space

`sweep` solves every *H* × *W* point and appends one line per point to a `.jsonl` file;
rerunning skips finished points. `maps` turns it into figures; `design_map.png`
shows *D* and the regime of each point:

* **target** (green): anomalous (*D* ≥ 0) with exactly one guided TE mode;
* **multimode** (red): anomalous but two or more guided TE modes;
* **normal** (grey): *D* < 0.

In [ ]:
# ---- Step 2: grid sweep and maps ---- #
grid = Path("results/demo_grid.jsonl")
tool("sweep", "--heights", "0.70:0.80:0.10", "--widths", "1.0:1.2:0.2", "--wl", WL,
     "--steps", STEPS, "--workers", 2, "--out", grid, *STACK)
tool("maps", "--data", grid, "--overwrite")

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/plots_demo_grid/design_map.png"))

## 3. Mesh convergence

*D* depends on the mesh. `convergence` solves the same points at several mesh densities;
pick the density beyond which *D* stops changing (within `--tol-pct`, 2 % by default).

In [ ]:
# ---- Step 3: mesh convergence ---- #
tool("convergence", "--points", "0.70x1.00", "--steps-list", "30,50,70", "--wl", WL,
     "--workers", 2, "--data", "results/demo_convergence.jsonl", "--overwrite", *STACK)

## 4. Finalist and fabrication tolerance

`sweep --points H×W --tol dW,dH` solves the finalist and its 3 × 3 neighbourhood
(*W* ± d*W*, *H* ± d*H*). `report` checks whether the whole neighbourhood stays in the
target region and writes `report.md` and `finalists.csv`. The first finalist is the
baseline the others are compared with.

In [ ]:
# ---- Step 4: finalist with a +/-50 nm width, +/-20 nm height tolerance ---- #
finalists = Path("results/demo_finalists.jsonl")
tool("sweep", "--points", "0.70x1.00", "--tol", "0.05,0.02", "--wl", WL, "--steps", STEPS,
     "--workers", 2, "--out", finalists, *STACK)
tool("report", "--straight", finalists, "--finalists", "0.70x1.00", "--tol", "0.05,0.02",
     "--out", "results/demo_report", "--overwrite")
# The full report with all columns: results/demo_report/report.md

## 5. Broadband dispersion

`broadband` solves the finalist over a wide band in one go and reports *D*(λ), the
zero-dispersion wavelengths and, for a ring of radius `--radius`, the integrated
dispersion *D*<sub>int</sub>(μ) with *D*₂ and *D*₃. Keep the band inside the range of the
material data.

In [ ]:
# ---- Step 5: D(lambda) and D_int for a 100 um ring ---- #
tool("broadband", "--points", "0.70x1.00", "--wl-min", round(WL - 0.25, 3),
     "--wl-max", round(WL + 0.25, 3), "--steps", STEPS, "--radius", 100,
     "--data", "results/demo_broadband.jsonl", "--overwrite", *STACK)